# 🎲 Gewinnwahrscheinlichkeihen - symbolisch

Dieses Programm berechnet für $p_0,\ldots,p_{m-1}$  **symbolisch** die Gewinnwahrscheinlichkeiten  
$$ P_A(V,W),\ P_B(V,W),\ P_U(V,W). $$
Zusätzlich werden für vorgegebene Felderwahrscheinlichkeiten die Gewinnwahrscheinlichkeiten **exakt** und **numerisch** ausgegeben.

---

### ✅ Voraussetzungen

- Es wird genau **ein Würfel** mit symbolischen ($p_0,\ldots,p_{m-1}$) und benutzerdefinierten Wahrscheinlichkeiten verwendet, die sich zu $1$ summieren.

---

### ⚙️ Eingaben

- `p_vals`: die benutzerdefinierten Wahrscheinlichkeiten für die einzelnen Felder
- `S`: Setzstrategie
- `T`: Setzstrategie

📌 **Vorsicht:** Es enstehen schnell "Monsterteme". Verwenden Sie nur kleine Chipanzahlen.

---
### 🧩 Modul `PropSetzstrategien.py`

Die Datei `PropSetzstrategien.py` enthält **alle zentralen Funktionen** zur Berechnung:

- Gewinnwahrscheinlichkeiten (`P_A`, `P_B`, `P_U`)
- Strategievergleiche
- Simulationen und Konfidenzintervalle
- Visualisierungen

> 📌 **Wichtig:** Diese Datei muss sich im **Hauptverzeichnis** befinden,  
> damit sie in den Notebooks importiert werden kann (z. B. `import PropSetzstrategien as ps`).

Man kann `PropSetzstrategien.py` als **Funktionssammlung** bezeichnen – sie wird von allen Jupyter-Notebooks gemeinsam genutzt.

---

### ▶️ So geht’s weiter

In der **nächsten Code-Zelle** befindet sich das zugehörige Python-Programm.  
Dort müssen lediglich folgende Eingaben angepasst werden:

- `p_val`: benuterdefinierte Wahrscheinlichkeiten für die Felder (z. B. `p = (1/2, 1/3, 1/6)`)
- `S`: Setzstrategie  (z. B. `S = (2, 0, 0)`)
- `T`: Setzstrategie  (z.B. `T = (0, 0, 1)`)


> 📌 Achte Sie darauf, dass `p`, `S` und `T` gleich lang sind!

---

### ▶️ Ausführen des Programms

- Klicken Sie in die Code-Zelle.
- Drücken Sie `Shift + Enter`, um die Berechnung zu starten.  
  Alternativ können Sie auch auf **▶ Run** oben in der Werkzeugleiste klicken.
- Über **Run → Run all cells** werden alle Zellen auf einmal ausgeführt.


📎 Viel Erfolg beim Experimentieren mit eigenen Strategien!

---

In [5]:
from functools import lru_cache
from fractions import Fraction
import sympy as sp

# ======== Eingabe ===================================
p_vals = [Fraction(1,2), Fraction(3,10), Fraction(2,10)]
S = (2,0,0)
T = (0,0,1)

# ============== Berechnungen ========================
def make_symbolic_PA(m):
    """
    Erzeugt P_A(S,T) als rekursive, symbolische Funktion in p0...p{m-1}.
    Rückgabe: (P_A_sym, p_symbols)
    """
    # Definiere p0, p1, ..., p{m-1}
    p_syms = sp.symbols(f"p0:{m}", positive=True)

    @lru_cache(maxsize=None)
    def P_A_sym(S, T):
        S, T = list(S), list(T)
        # Basisfälle
        if sum(S)==0 and sum(T)==0:
            return sp.Integer(0)
        if sum(S)==0:
            return sp.Integer(1)
        if sum(T)==0:
            return sp.Integer(0)
        # Normierung nur über Felder mit Chips
        s = sum(p_syms[i] for i in range(m) if S[i] or T[i])
        expr = sp.Integer(0)
        for i in range(m):
            if not (S[i] or T[i]):
                continue
            S2 = S.copy(); T2 = T.copy()
            if S2[i]>0: S2[i] -= 1
            if T2[i]>0: T2[i] -= 1
            expr += (p_syms[i]/s) * P_A_sym(tuple(S2), tuple(T2))
        return sp.simplify(expr)

    return P_A_sym, p_syms

def compute_symbolic_outcomes(S, T, m):
    """
    Gibt die symbolischen Ausdrücke (P_A, P_B, P_U) in p0...p{m-1} zurück.
    """
    P_A_sym, p_syms = make_symbolic_PA(m)
    PA_expr = P_A_sym(tuple(S), tuple(T))
    PB_expr = P_A_sym(tuple(T), tuple(S))
    PU_expr = sp.simplify(1 - PA_expr - PB_expr)
    return (PA_expr, PB_expr, PU_expr), p_syms

def compute_numeric_outcomes(S, T, p_vals):
    """
    Evaluiert symbolisch erzeugte Ausdrücke numerisch:
    - als sympy.Rational
    - als Python Fraction
    - als float
    Rückgabe-Dict mit drei Tupeln.
    """
    m = len(p_vals)
    # 1) Symbolisch
    (PA_expr, PB_expr, PU_expr), p_syms = compute_symbolic_outcomes(S, T, m)

    # 2) Substitution in sympy.Rational
    subs = {p_syms[i]: sp.Rational(p_vals[i]) for i in range(m)}
    PA_rat = sp.simplify(PA_expr.subs(subs))
    PB_rat = sp.simplify(PB_expr.subs(subs))
    PU_rat = sp.simplify(PU_expr.subs(subs))

    # 3) Umwandlung in Python Fraction
    PA_frac = Fraction(PA_rat.p, PA_rat.q)
    PB_frac = Fraction(PB_rat.p, PB_rat.q)
    PU_frac = Fraction(PU_rat.p, PU_rat.q)

    # 4) Fließkomma-Werte
    PA_f = float(PA_rat)
    PB_f = float(PB_rat)
    PU_f = float(PU_rat)

    return {
        'symbolic': (PA_expr, PB_expr, PU_expr),
        'rational': (PA_frac, PB_frac, PU_frac),
        'float':    (PA_f, PB_f, PU_f)
    }

out = compute_numeric_outcomes(S, T, p_vals)

# ============= Ausgabe ======================
for mode in ('symbolic','rational','float'):
    print(f"\n=== {mode.upper()} ===")
    results = out[mode]
    labels  = (f'P_A({S},{T})',f'P_B({S},{T})',f'P_U({S},{T})')
    for lbl, val in zip(labels, results):
        if mode == 'symbolic':
            print(f"\n{lbl} =\n", end=" ")
            sp.pprint(val, use_unicode=True)
        else:
            print(f"{lbl} = {val}")


=== SYMBOLIC ===

P_A((2, 0, 0),(0, 0, 1)) =
      2    
   p₀     
──────────
         2
(p₀ + p₂) 

P_B((2, 0, 0),(0, 0, 1)) =
 p₂⋅(2⋅p₀ + p₂)
──────────────
           2  
  (p₀ + p₂)   

P_U((2, 0, 0),(0, 0, 1)) =
 0

=== RATIONAL ===
P_A((2, 0, 0),(0, 0, 1)) = 25/49
P_B((2, 0, 0),(0, 0, 1)) = 24/49
P_U((2, 0, 0),(0, 0, 1)) = 0

=== FLOAT ===
P_A((2, 0, 0),(0, 0, 1)) = 0.5102040816326531
P_B((2, 0, 0),(0, 0, 1)) = 0.4897959183673469
P_U((2, 0, 0),(0, 0, 1)) = 0.0
